# Installations

In [1]:
%load_ext autoreload
%autoreload 2

ModuleNotFoundError: No module named 'imp'

In [2]:
%pip install -q mlflow databricks-sdk optuna

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.7/49.7 kB 3.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.5/50.5 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 96.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 102.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 73.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 969.1/969.1 kB 53.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.6/425.6 kB 26.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 264.7/264.7 kB 26.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 62.4 MB/s eta 0:00:00ta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 148.8/148.8 kB 16.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━

# Setup

In [3]:
try:
  import google.colab
  RUN_ENV = 'colab'
except ImportError:
  RUN_ENV = 'local'

In [4]:
COLAB_KAGGLE_UTILS_PATH = '/content/drive/MyDrive/KaggleData'
LOCAL_KAGGLE_UTILS_PATH = '/home/sameera/Projects'

In [6]:
import sys

if RUN_ENV == 'colab':
    sys.path.append(COLAB_KAGGLE_UTILS_PATH)
    dataPath = '/content/drive/MyDrive/KaggleData/s6e7/'
    from google.colab import drive
    drive.mount('/content/drive')
else:
    sys.path.append(LOCAL_KAGGLE_UTILS_PATH)
    dataPath = '/home/sameera/Projects/Kaggle/data/KaggleData/s6e7/'

Mounted at /content/drive


In [27]:
import pandas as pd

from sklearn.preprocessing import OrdinalEncoder, LabelEncoder, OneHotEncoder
from sklearn.impute import SimpleImputer

# Models CPU
from sklearn.ensemble import RandomForestClassifier

#GPU
# from cuml.ensemble import RandomForestClassifier
import cupy as cp

# Tuning
import optuna
# from optuna.integration import LightGBMPruningCallback

import mlflow
import os
import json

# Metrics 
from sklearn.metrics import balanced_accuracy_score

# Custom utilities
from kaggle_utils.data.preprocessor import PreprocessorFactory
from kaggle_utils.models.learner import Learner
from kaggle_utils.models.gpu_tuner import GPUTuner
from kaggle_utils.models.tree_model_diagnostics import TreeModelDiagnostics

In [34]:
import numpy as np
import pandas as pd
pd.set_option('display.max_columns', None)

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

# Preprocessing
from sklearn.preprocessing import OrdinalEncoder, LabelEncoder, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer


from imblearn.over_sampling import RandomOverSampler

# Model Selection
from sklearn.model_selection import StratifiedKFold 

# Models CPU
from sklearn.ensemble import RandomForestClassifier

#GPU
# from cuml.ensemble import RandomForestClassifier
# import cupy as cp

# Tuning
import optuna
# from optuna.integration import LightGBMPruningCallback

import mlflow
import os
import json

# Metrics 
from sklearn.metrics import balanced_accuracy_score

import warnings
warnings.filterwarnings('ignore')

In [8]:
train = pd.read_csv(dataPath + 'train.csv', low_memory=False)
test = pd.read_csv(dataPath + 'test.csv', low_memory=False)
sample_submission = pd.read_csv(dataPath + 'sample_submission.csv', index_col='id', low_memory=False)

# Data Glance

In [6]:
train.head()

,id,health_condition,sleep_duration,heart_rate,bmi,calorie_expenditure,step_count,exercise_duration,water_intake,diet_type,stress_level,sleep_quality,physical_activity_level,smoking_alcohol,gender
0,0,unhealthy,5.22,70.6,25.66,2174.0,1326.0,19.8,1.86,veg,high,average,sedentary,yes,female
1,1,at-risk,5.53,71.3,25.84,1966.0,9891.0,49.9,1.26,non-veg,low,average,moderate,yes,other
2,2,unhealthy,5.29,75.4,24.54,2688.0,14216.0,38.1,1.60,veg,high,poor,active,yes,male
3,3,unhealthy,4.70,77.2,23.13,2630.0,7174.0,59.9,2.02,veg,high,average,active,occasional,female
4,4,at-risk,7.23,73.4,28.44,2560.0,6584.0,46.0,2.25,veg,NaN,average,sedentary,NaN,male


In [7]:
test.head()

,id,sleep_duration,heart_rate,bmi,calorie_expenditure,step_count,exercise_duration,water_intake,diet_type,stress_level,sleep_quality,physical_activity_level,smoking_alcohol,gender
0,690088,5.35,64.9,23.48,2745.0,14167.0,59.5,1.86,veg,high,poor,active,occasional,male
1,690089,NaN,83.1,22.42,1773.0,6801.0,24.5,2.40,balanced,high,poor,sedentary,yes,other
2,690090,6.68,59.7,24.14,3040.0,13250.0,48.5,2.76,balanced,medium,poor,active,no,NaN
3,690091,7.13,78.5,26.26,2494.0,6331.0,56.9,2.34,veg,low,good,moderate,yes,other
4,690092,5.49,77.7,23.29,1828.0,13894.0,39.4,2.45,veg,high,average,active,occasional,other


In [9]:
sample_submission.head()

,health_condition
id,
690088,at-risk
690089,at-risk
690090,at-risk
690091,at-risk
690092,at-risk


In [10]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 690088 entries, 0 to 690087
Data columns (total 15 columns):
 #   Column                   Non-Null Count   Dtype  
---  ------                   --------------   -----  
 0   id                       690088 non-null  int64  
 1   health_condition         690088 non-null  object 
 2   sleep_duration           614089 non-null  float64
 3   heart_rate               682255 non-null  float64
 4   bmi                      676190 non-null  float64
 5   calorie_expenditure      637235 non-null  float64
 6   step_count               676172 non-null  float64
 7   exercise_duration        683187 non-null  float64
 8   water_intake             646611 non-null  float64
 9   diet_type                683187 non-null  object 
 10  stress_level             607277 non-null  object 
 11  sleep_quality            631757 non-null  object 
 12  physical_activity_level  653467 non-null  object 
 13  smoking_alcohol          661506 non-null  object 
 14  gend

In [11]:
test.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 295753 entries, 0 to 295752
Data columns (total 14 columns):
 #   Column                   Non-Null Count   Dtype  
---  ------                   --------------   -----  
 0   id                       295753 non-null  int64  
 1   sleep_duration           263182 non-null  float64
 2   heart_rate               292396 non-null  float64
 3   bmi                      289797 non-null  float64
 4   calorie_expenditure      273101 non-null  float64
 5   step_count               289789 non-null  float64
 6   exercise_duration        292795 non-null  float64
 7   water_intake             277120 non-null  float64
 8   diet_type                292795 non-null  object 
 9   stress_level             260263 non-null  object 
 10  sleep_quality            270754 non-null  object 
 11  physical_activity_level  280058 non-null  object 
 12  smoking_alcohol          283504 non-null  object 
 13  gender                   286593 non-null  object 
dtypes: f

# Features

In [9]:
conts = ['sleep_duration', 'heart_rate', 'bmi', 'calorie_expenditure','step_count', 'exercise_duration', 'water_intake']
cats = ['diet_type', 'stress_level', 'sleep_quality', 'physical_activity_level','smoking_alcohol', 'gender']

target = 'health_condition'

# Preprocessing

## Encoding

In [10]:
catOrders = {
  "stress_level": ['low', 'medium', 'high'],
  "sleep_quality": ['poor', 'average', 'good'],
  "physical_activity": ['sedentary','moderate', 'active'],
  "smoking_alcohol": ['no', 'occasional', 'yes']
}

In [14]:
le  = LabelEncoder()
train[target] = le.fit_transform(train[target])

In [15]:
feature_cols = conts + cats

### Testing Preprocessor

In [19]:
# Create the factory once
factory = PreprocessorFactory(conts, cats, catOrders)
# selected_features = ['sleep_duration', 'stress_level', 'physical_activity_level', 'bmi']

dynamic_preprocessor = factory.build(feature_cols, cont_imputer=SimpleImputer(strategy='constant', fill_value=999999))

In [20]:
t = dynamic_preprocessor.fit_transform(train[feature_cols])
print(t.head())

   continuous__sleep_duration  continuous__heart_rate  continuous__bmi  \
0                        5.22                    70.6            25.66   
1                        5.53                    71.3            25.84   
2                        5.29                    75.4            24.54   
3                        4.70                    77.2            23.13   
4                        7.23                    73.4            28.44   

   continuous__calorie_expenditure  continuous__step_count  \
0                           2174.0                  1326.0   
1                           1966.0                  9891.0   
2                           2688.0                 14216.0   
3                           2630.0                  7174.0   
4                           2560.0                  6584.0   

   continuous__exercise_duration  continuous__water_intake  \
0                           19.8                      1.86   
1                           49.9                      1.26

# Modelling

## setup MLflow

In [23]:
with open('/content/drive/MyDrive/KaggleData/mlflow/databricks.json', 'r') as f:
  config = json.load(f)

os.environ["DATABRICKS_HOST"] = config.get("DATABRICKS_HOST", "")
os.environ["DATABRICKS_TOKEN"] = config.get("DATABRICKS_TOKEN", "")
os.environ["MLFLOW_TRACKING_URI"] = "databricks"

In [24]:
user_email = "sameeranc11@gmail.com"
experiment_path = f"/Users/{user_email}/Kaggle_s6e7_Comp"
# mlflow.sklearn.autolog(log_models=False) #removing this because  it's creating conflicts with manual logging.
mlflow.set_experiment(experiment_path)

If you are using MLflow Tracing, you can migrate your traces to Unity Catalog for unlimited storage, fine-grained access controls, and queryability from notebooks, SQL, and dashboards. Learn more: https://docs.databricks.com/aws/en/mlflow3/genai/tracing/migrate-traces-to-uc


<Experiment: artifact_location='dbfs:/databricks/mlflow-tracking/1584872737403196', creation_time=1784457247063, effective_trace_archival_retention=None, experiment_id='1584872737403196', last_update_time=1784994263233, lifecycle_stage='active', name='/Users/sameeranc11@gmail.com/Kaggle_s6e7_Comp', tags={'mlflow.experiment.sourceName': '/Users/sameeranc11@gmail.com/Kaggle_s6e7_Comp',
 'mlflow.experimentKind': 'custom_model_development',
 'mlflow.experimentType': 'MLFLOW_EXPERIMENT',
 'mlflow.ownerEmail': 'sameeranc11@gmail.com',
 'mlflow.ownerId': '75150582631674'}, trace_location=None, workspace='default'>

## setup kaggle

In [25]:
with open('/content/drive/MyDrive/KaggleData/kaggle.json', 'r') as f:
  config = json.load(f)

os.environ["KAGGLE_API_TOKEN"] = config.get("KAGGLE_API_TOKEN", "")

In [30]:
def submit_preds(model, target, fileName):   
  final_string_predictions = le.inverse_transform(model.final_test_predictions)
  sample_submission[target] = final_string_predictions
  sample_submission.to_csv(dataPath + fileName)

## Basic RF

In [ ]:
%%time
# factory = PreprocessorFactory(conts, cats, catOrders)
# dynamic_preprocessor = factory.build(feature_cols)

rf_model = Learner(train, test, target, RandomForestClassifier, feature_cols, 'RF_Baseline_ClassBalanced_Ordinal', balanced_accuracy_score, preprocessor=dynamic_preprocessor)
rf_model.fit(trainingMode='standard', params={'class_weight': 'balanced', 'n_jobs': -1}, ploting={"fi":True, "score_vs_trees":True})

Training fold 1...
Fold 1 ==> OOF score: 0.8645287941391326
Training fold 2...
Fold 2 ==> OOF score: 0.8694700333056669
Training fold 3...
Fold 3 ==> OOF score: 0.8683446480390747
Training fold 4...
Fold 4 ==> OOF score: 0.8637652985922354
Training fold 5...
Fold 5 ==> OOF score: 0.8655419253729525
CV Mean Accuracy: 0.8663 | Std: 0.0022
---------------------------------------------------------------
🏃 View run RF_Baseline_ClassBalanced_Ordinal_standard at: https://dbc-36754e4d-ad0b.cloud.databricks.com/ml/experiments/1584872737403196/runs/874cedeef9184875bb56fac64a207512
🧪 View experiment at: https://dbc-36754e4d-ad0b.cloud.databricks.com/ml/experiments/1584872737403196
CPU times: user 19min 30s, sys: 3.02 s, total: 19min 33s
Wall time: 11min 17s


np.float64(0.8663301398898124)

In [29]:
rf_model = Learner(train, test, target, RandomForestClassifier, feature_cols, 'RF_Baseline_ClassBalanced_OneHot', balanced_accuracy_score, preprocessor=basic_preprocessor)
rf_model.fit(trainingMode='standard', params={'class_weight': 'balanced', 'n_jobs': -1}, ploting={"fi":True, "score_vs_trees":True})

Training fold 1...
Fold 1 ==> OOF score: 0.8627837512231104
Training fold 2...
Fold 2 ==> OOF score: 0.8670132338309856
Training fold 3...
Fold 3 ==> OOF score: 0.8661038928220633
Training fold 4...
Fold 4 ==> OOF score: 0.8613287524034202
Training fold 5...
Fold 5 ==> OOF score: 0.863438575405281
CV Mean Accuracy: 0.8641 | Std: 0.0021
---------------------------------------------------------------
🏃 View run RF_Baseline_ClassBalanced_OneHot_standard at: https://dbc-36754e4d-ad0b.cloud.databricks.com/ml/experiments/1584872737403196/runs/40b7231ba3ca42089ceb3fb659ee2f28
🧪 View experiment at: https://dbc-36754e4d-ad0b.cloud.databricks.com/ml/experiments/1584872737403196


np.float64(0.8641336411369721)

In [31]:
rf_model = Learner(train, test, target, RandomForestClassifier, feature_cols, 'RF_Baseline_ClassBalanced_OneHot', balanced_accuracy_score, preprocessor=basic_preprocessor)
rf_model.fit(trainingMode='standard', params={'class_weight': 'balanced', 'n_jobs': -1, 'n_estimators': 100, 'max_depth': 12}, ploting={"fi":True, "score_vs_trees":True})

Training fold 1...
Fold 1 ==> OOF score: 0.9454945365994832
Training fold 2...
Fold 2 ==> OOF score: 0.9467022400829238
Training fold 3...
Fold 3 ==> OOF score: 0.9449938210904033
Training fold 4...
Fold 4 ==> OOF score: 0.9458634152271675
Training fold 5...
Fold 5 ==> OOF score: 0.9437881262931519
CV Mean Accuracy: 0.9454 | Std: 0.0010
---------------------------------------------------------------
🏃 View run RF_Baseline_ClassBalanced_OneHot_standard at: https://dbc-36754e4d-ad0b.cloud.databricks.com/ml/experiments/1584872737403196/runs/2b1ddc9c2ca64b17946036c266a354ae
🧪 View experiment at: https://dbc-36754e4d-ad0b.cloud.databricks.com/ml/experiments/1584872737403196


np.float64(0.9453684278586258)

In [21]:
factory = PreprocessorFactory(conts, cats, catOrders)
selected_features = ['sleep_duration', 'stress_level', 'physical_activity_level', 'bmi', 'exercise_duration', 'step_count']
dynamic_preprocessor = factory.build(selected_features)

In [23]:
%%time
rf_model = Learner(train, test, target, RandomForestClassifier, selected_features, 'RF_Baseline_ClassBalanced_Ordinal_topF', balanced_accuracy_score, preprocessor=dynamic_preprocessor)
rf_model.fit(trainingMode='standard', params={'class_weight': 'balanced', 'n_jobs': -1, 'n_estimators': 100, 'max_depth': 12}, ploting={"fi":True, "score_vs_trees":True})

Training fold 1...
Fold 1 ==> OOF score: 0.9475552868434188
Training fold 2...
Fold 2 ==> OOF score: 0.9497735122604594
Training fold 3...
Fold 3 ==> OOF score: 0.9476084507403869
Training fold 4...
Fold 4 ==> OOF score: 0.9473032038342236
Training fold 5...
Fold 5 ==> OOF score: 0.9454190215793123
CV Mean Accuracy: 0.9475 | Std: 0.0014
---------------------------------------------------------------
🏃 View run RF_Baseline_ClassBalanced_Ordinal_topF_standard at: https://dbc-36754e4d-ad0b.cloud.databricks.com/ml/experiments/1584872737403196/runs/1f47766cf3f4485194e0bf61c1451350
🧪 View experiment at: https://dbc-36754e4d-ad0b.cloud.databricks.com/ml/experiments/1584872737403196
CPU times: user 12min 23s, sys: 1.94 s, total: 12min 25s
Wall time: 8min 8s


np.float64(0.9475318950515602)

In [ ]:
submit_preds(rf_model, target, 'RF_Baseline_ClassBalanced_Ordinal_topF.csv')   

In [35]:
!kaggle competitions submit -c playground-series-s6e7 -f /content/drive/MyDrive/KaggleData/s6e7/RF_Base_Cont_LV_Imputed.csv  -m "RF_Base_Cont_LV_Imputed"

100% 4.21M/4.21M [00:00<00:00, 6.27MB/s]
Successfully submitted to Predicting Student Health Risk

# RF Tuning

In [24]:
def rf_param_space(trial):
    """Defines the hyperparameter space for RandomForestClassifier."""
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 200, 10),
        'max_depth': trial.suggest_int('max_depth', 7, 15, 1),
        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 2, 15, 1),
        'max_features':trial.suggest_int('max_features',2,10,1)
    }
    return params

In [41]:
factory = PreprocessorFactory(conts, cats, catOrders)
selected_features = ['sleep_duration', 'stress_level', 'physical_activity_level', 'bmi', 'exercise_duration', 'step_count', 'sleep_quality']
dynamic_preprocessor = factory.build(selected_features)

In [30]:
rf_tuner = Learner(train, test, target, RandomForestClassifier, feature_cols, 'RF_Tuner_NewDepth', balanced_accuracy_score, preprocessor=dynamic_preprocessor)
best_params = rf_tuner.fast_gpu_tune(study_name='RF_GPU_Tuning_NewDepth', n_trials=50, get_params_func=rf_param_space, db_path='/content/drive/MyDrive/KaggleData/s6e7/rf_gpu_tuning_study.db')

[I 2026-07-25 13:50:17,080] Using an existing study with name 'RF_GPU_Tuning_NewDepth' instead of creating a new one.


  0%|          | 0/50 [00:00<?, ?it/s]

🏃 View run Trial_3 at: https://dbc-36754e4d-ad0b.cloud.databricks.com/ml/experiments/1584872737403196/runs/48fd6925c14a44efadd6be0f40966d43
🧪 View experiment at: https://dbc-36754e4d-ad0b.cloud.databricks.com/ml/experiments/1584872737403196
[I 2026-07-25 13:51:19,392] Trial 3 finished with value: 0.9457939871988785 and parameters: {'n_estimators': 150, 'max_depth': 7, 'min_samples_leaf': 9, 'max_features': 8}. Best is trial 3 with value: 0.9457939871988785.
🏃 View run Trial_4 at: https://dbc-36754e4d-ad0b.cloud.databricks.com/ml/experiments/1584872737403196/runs/4b75497644484d9f9136bcec299d6b86
🧪 View experiment at: https://dbc-36754e4d-ad0b.cloud.databricks.com/ml/experiments/1584872737403196
[I 2026-07-25 13:52:04,691] Trial 4 finished with value: 0.9455648386926765 and parameters: {'n_estimators': 130, 'max_depth': 8, 'min_samples_leaf': 10, 'max_features': 4}. Best is trial 3 with value: 0.9457939871988785.
🏃 View run Trial_5 at: https://dbc-36754e4d-ad0b.cloud.databricks.com/ml/ex

In [31]:
best_params

{'n_estimators': 190,
 'max_depth': 10,
 'min_samples_leaf': 12,
 'max_features': 8}

In [42]:
%%time
rf_model = Learner(train, test, target, RandomForestClassifier, selected_features, 'RF_Tuned_ClassBalanced_Ordinal_TopF', balanced_accuracy_score, preprocessor=dynamic_preprocessor)
rf_model.fit(trainingMode='standard', params={'n_estimators': 190,
 'max_depth': 10,
 'min_samples_leaf': 12,
 'class_weight': 'balanced'}, ploting={"fi":True, "score_vs_trees":True})

Training fold 1...
Fold 1 ==> OOF score: 0.9472545081634672
Training fold 2...
Fold 2 ==> OOF score: 0.9492369185879115
Training fold 3...
Fold 3 ==> OOF score: 0.9469208439565916
Training fold 4...
Fold 4 ==> OOF score: 0.9473155913998857
Training fold 5...
Fold 5 ==> OOF score: 0.9445057639156843
CV Mean Accuracy: 0.9470 | Std: 0.0015
---------------------------------------------------------------
🏃 View run RF_Tuned_ClassBalanced_Ordinal_TopF_standard at: https://dbc-36754e4d-ad0b.cloud.databricks.com/ml/experiments/1584872737403196/runs/928aaf6114a4433eb797e9912e016d98
🧪 View experiment at: https://dbc-36754e4d-ad0b.cloud.databricks.com/ml/experiments/1584872737403196
CPU times: user 10min 30s, sys: 0 ns, total: 10min 30s
Wall time: 10min 51s


np.float64(0.9470467252047081)

In [43]:
submit_preds(rf_model, target, 'RF_Tuned_ClassBalanced_Ordinal_TopF.csv')   

# RF Kaggle Utils Testing - Conts missing large value imputing

In [26]:
factory = PreprocessorFactory(conts, cats, catOrders)
dynamic_preprocessor = factory.build(feature_cols, cont_imputer=SimpleImputer(strategy='constant', fill_value=999999))

In [33]:
rf_model = Learner(train, test, target, RandomForestClassifier, feature_cols, 'RF_Base_Cont_LV_Imputed', balanced_accuracy_score, preprocessor=dynamic_preprocessor)
rf_model.fit(trainingMode='standard', params={'max_depth': 10,
 'class_weight': 'balanced'}, plotting={"fi":True, "score_vs_trees":True, "tree_depth":True})

Training fold 1...
Fold 1 ==> OOF score: 0.9465739019266984


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but DecisionTreeClassifier was fitted without feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but DecisionTreeClassifier was fitted without feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but DecisionTreeClassifier was fitted without feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but DecisionTreeClassifier was fitted without feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but DecisionTreeClassifier was fitted without feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2732: U

Training fold 2...
Fold 2 ==> OOF score: 0.9486902337073065


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but DecisionTreeClassifier was fitted without feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but DecisionTreeClassifier was fitted without feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but DecisionTreeClassifier was fitted without feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but DecisionTreeClassifier was fitted without feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but DecisionTreeClassifier was fitted without feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2732: U

Training fold 3...
Fold 3 ==> OOF score: 0.946591418562071


...

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but DecisionTreeClassifier was fitted without feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but DecisionTreeClassifier was fitted without feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but DecisionTreeClassifier was fitted without feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but DecisionTreeClassifier was fitted without feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but DecisionTreeClassifier was fitted without feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2732: U

Training fold 5...
Fold 5 ==> OOF score: 0.9437099994612757


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but DecisionTreeClassifier was fitted without feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but DecisionTreeClassifier was fitted without feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but DecisionTreeClassifier was fitted without feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but DecisionTreeClassifier was fitted without feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but DecisionTreeClassifier was fitted without feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2732: U

CV Mean Accuracy: 0.9464 | Std: 0.0016
---------------------------------------------------------------
🏃 View run RF_Base_Cont_LV_Imputed_standard at: https://dbc-36754e4d-ad0b.cloud.databricks.com/ml/experiments/1584872737403196/runs/ec72835059a74bdb87e209056c9260c4
🧪 View experiment at: https://dbc-36754e4d-ad0b.cloud.databricks.com/ml/experiments/1584872737403196


np.float64(0.9463627948236508)

In [34]:
submit_preds(rf_model, target, 'RF_Base_Cont_LV_Imputed.csv')   